# 00 — Setup, 10K BPE cache, and correctness gates

This notebook prepares the shared TinyStories token cache and proves the reversible implementation before any expensive 50M-token run. The custom **10,000-token BPE** is trained on TinyStories so the ~20M parameter budget goes primarily into transformer depth rather than a 50K-token embedding table.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = "https://github.com/JoeIndyGit/era-v5-session-13-reversible-llm-lab.git"
REPO_DIR = Path("/content/era-v5-session-13-reversible-llm-lab")
if not (Path.cwd() / "src").exists():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, str(Path.cwd()))
print("repo root:", Path.cwd())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    candidates = [p.parent for p in Path("/content").glob("**/src") if p.is_dir()]
    if candidates:
        ROOT = candidates[0]
        os.chdir(ROOT)
print("repo root:", Path.cwd())
sys.path.insert(0, str(Path.cwd()))

In [ ]:
from src.utils import device_info
device_info()

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/validate.py'], check=True)


In [ ]:
from src.data import prepare_tinystories_cache
meta = prepare_tinystories_cache(
    out_dir="data",
    train_tokens=52_000_000,
    val_tokens=1_000_000,
    vocab_size=10_000,
    tokenizer_training_stories=100_000,
)
meta

## Gates that must pass

- exact parameter count: **20,000,768**;
- >85% of parameters in transformer blocks;
- baseline/reversible initial tensors identical;
- custom reversible backward matches naive autograd;
- reversible round-trip error is within tolerance;
- final partial batch can account for an exact token budget;
- LR schedule endpoints are correct;
- 52M/1M token caches and SHA-256 hashes exist.

If any gate fails, do not start the long runs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/capture_environment.py'], check=True)


Environment evidence is saved to `results/environment.txt`. The next cell validates all 22 layers at context 256 with the actual GPU precision, after three real optimizer updates.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/validate_gpu.py"], check=True)
